<a href="https://colab.research.google.com/github/JSJeong-me/GPT-Web/blob/main/303%20langchain_doc_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

공식 LangChain 문서 MCP 서버(`[https://docs.langchain.com/mcp](https://docs.langchain.com/mcp)`)는 HTTP 엔드포인트로 제공됩니다. MCP 도구를 LangChain 에이전트와 연결할 때는 비동기(async) 방식으로 도구를 불러와 `create_deep_agent`의 `tools` 매개변수에 전달해야 합니다.

환경에 맞춰 사용할 수 있도록 **최신 내장 어댑터(`langchain[mcp]`)** 방식과 **`langchain-mcp-adapters`** 방식을 기준으로 작성한 코드입니다.

---

### 방법 1. 최신 `MCPAdapter` 사용 (`langchain[mcp]`)

```bash
pip install -U "langchain[mcp]"

```

In [ ]:
!pip install -U "langchain[mcp]" "langchain-mcp-adapters>=0.2.0" "mcp>=1.2.0" langchain-openai langgraph nest_asyncio

In [2]:
from google.colab import userdata
import os

# 'OPENAI_API_KEY' 라는 이름으로 저장된 보안 비밀을 가져옵니다.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("OPENAI_API_KEY가 환경 변수로 설정되었습니다.")

OPENAI_API_KEY가 환경 변수로 설정되었습니다.


In [11]:
import asyncio
import sys

# 1. 특정 호환 버전으로 강제 재설치하여 ImportError 해결
!{sys.executable} -m pip install --force-reinstall "mcp==1.2.0" "langchain-mcp-adapters==0.2.0" "langchain-core>=1.6.0" langchain-openai langgraph

import nest_asyncio
nest_asyncio.apply()

# 2. 필수 모듈 임포트
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"

async def main():
    # 3. MCP 클라이언트를 통해 도구 연결
    async with MultiServerMCPClient(
        {"langchain-docs": {"url": "https://docs.langchain.com/mcp", "transport": "http"}}
    ) as client:
        tools = await client.get_tools()

        # 4. LLM 설정
        model = ChatOpenAI(model="gpt-4o")

        # 5. ReAct 에이전트 생성
        agent = create_react_agent(
            model,
            tools=tools,
            state_modifier="You are a helpful assistant using MCP tools to answer questions about LangChain."
        )

        # 6. 실행
        inputs = {"messages": [HumanMessage(content=EXAMPLE_QUERY)]}
        result = await agent.ainvoke(inputs)
        print(result["messages"][-1].content)

if __name__ == "__main__":
    try:
        # Colab/Jupyter 환경에 따른 루프 처리
        try:
            loop = asyncio.get_running_loop()
        except RuntimeError:
            loop = None

        if loop and loop.is_running():
            asyncio.create_task(main())
        else:
            asyncio.run(main())
    except Exception as e:
        print(f"Error during execution: {e}")

  Using cached mcp-2.2.0-py3-none-any.whl.metadata (7.8 kB)


ImportError: cannot import name 'RequestContext' from 'mcp.shared.context' (/usr/local/lib/python3.13/dist-packages/mcp/shared/context.py)

---

### 방법 2. `MultiServerMCPClient` 사용 (`langchain-mcp-adapters`)

```bash
pip install -U langchain-mcp-adapters

```

In [ ]:
import asyncio
from deepagents import create_deep_agent
from langchain.messages import HumanMessage
from langchain_mcp_adapters.client import MultiServerMCPClient

EXAMPLE_QUERY = "How do I stream intermediate tool results from a subagent?"

async def main():
    # 1. MCP 클라이언트 설정 및 도구 로드
    client = MultiServerMCPClient(
        {
            "langchain-docs": {
                "url": "https://docs.langchain.com/mcp",
                "transport": "http",
            }
        }
    )
    tools = await client.get_tools()

    # 2. 에이전트 생성 시 도구 연결
    agent = create_deep_agent(
        model="openai:gpt-5.5",
        tools=tools,
        system_prompt=(
            "You are a helpful LangChain documentation assistant. "
            "Use the provided documentation tools to look up APIs, patterns, "
            "and accurately answer user questions."
        ),
    )

    # 3. 실행
    result = await agent.ainvoke(
        {"messages": [HumanMessage(content=EXAMPLE_QUERY)]}
    )

    print(result["messages"][-1].text)

if __name__ == "__main__":
    asyncio.run(main())

---

### 주요 변경 사항

1. **`tools` 주입**: 빈 리스트 `tools=[]` 대신 MCP 서버에서 검색/조회용으로 노출한 도구 목록(`tools`)을 전달하여 LLM이 답변 생성 시 최신 문서를 직접 조회합니다.
2. **비동기 처리 (`asyncio` / `ainvoke`)**: MCP 프로토콜 통신(도구 목록 탐색 및 원격 RPC 호출) 특성상 비동기로 핸드셰이크가 이루어지므로 `async/await` 및 `ainvoke`를 사용합니다.
3. **시스템 프롬프트 보강**: 에이전트가 자체 지식에만 의존하지 않고 전달된 MCP 도구를 적극 활용하도록 유도 문구를 추가했습니다.